# Outlier Analysis

Adam L. Johnson, September 2021

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import pandas as pd

import nibabel as nib
import matplotlib.pyplot as plt
import cv2 as cv
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed

repo_path = Path('~/ukbb-pulmonary-artery/DeepCMR')
assert repo_path.is_dir(), "The repo_path must be set to the DeepCMR root in order to find custom modules"
if not str(repo_path.resolve()) in sys.path: sys.path.append(str(repo_path.resolve()))

from utils import geometry, strings as dstr, visualizer, signal_pro

___
## Define paths

In [2]:
## ================================== EDIT THESE ==================================
## Location of our project folder
PROJ_ROOT = Path('{deepcmr_data_root}/nnUNET/')
# PROJ_ROOT = repo_path

## 2. Relative path to compiled, predicted niftis
##    (created by notebooks/results/1_nnUNet_results.ipynb)
originals_path = PROJ_ROOT.joinpath('../cmr_lvot_20212_niftis/')
# originals_path = PROJ_ROOT.joinpath('sample_data/original_niftis')

## 2. Relative path to compiled, predicted niftis
##    (created by notebooks/results/1_nnUNet_results.ipynb)
prediction_path = PROJ_ROOT.joinpath('../predicted_niftis_nnUNet/Task618_UKBBPulmonaryArtery/cmr_lvot_20212/')
# prediction_path = PROJ_ROOT.joinpath('sample_data/predicted_niftis')

## 4. Output folder
##    (where to put .csv.gz results)
output_path = repo_path.joinpath('results/nnUNet/Task618_UKBBPulmonaryArtery/cmr_lvot_20212')
# output_path = PROJ_ROOT.joinpath('sample_data/results')
## ================================================================================

Read the file index and data

In [6]:
df_file_index_path = output_path.joinpath('df_fileindex.csv.gz')
print('Reading file index from ' + str(df_file_index_path))
df_file_index = pd.read_csv(df_file_index_path)

# df_dynamics_path = output_path.joinpath('df_dynamics.csv.gz')
# print('Reading dynamics from ' + str(df_dynamics_path))
# df_dynamics= pd.read_csv( df_dynamics_path )

df_geometry_path = output_path.joinpath('df_geometry.csv.xz')
print('Reading geometries from ' + str(df_geometry_path))
df_geometry = pd.read_csv(df_geometry_path)

Reading file index from ~/ukbb-pulmonary-artery/DeepCMR/results/nnUNet/Task618_UKBBPulmonaryArtery/cmr_lvot_20212/df_fileindex.csv.gz
Reading geometries from ~/ukbb-pulmonary-artery/DeepCMR/results/nnUNet/Task618_UKBBPulmonaryArtery/cmr_lvot_20212/df_geometry.csv.xz


---
## Display outliers

Use the list of outliers generated in R to make image previews. Save all of these images as separate files.

In [16]:
df_outliers = pd.read_csv( output_path.joinpath("outlier_candidates_reviewed_09.08.21.csv") )

df_outliers = df_outliers[~df_outliers.Unusable]

for name in tqdm(df_outliers.sample(5).Name):
    row = df_file_index.loc[df_file_index.Name == name].iloc[0]

    # outfile = output_path.joinpath(row.Name + '.png')
    # fig = visualizer.makefig_overview(row.OriginalPath, row.LabelPath, row.Name, margin=20, frame_interval=5, ellipse_show=False, plt_show=False)
    # fig.tight_layout()
    # fig.savefig(outfile)
    # plt.close(fig)

    try:
        anim = visualizer.make_mask_animation(row.OriginalPath, row.LabelPath, row.Name,
            output_path.joinpath("mp4_outliers/" + row.Name + ".mp4"),
            margin=20,
            ffmpeg='~/apps/bin/ffmpeg',
            area_plot=True)
    except:
        pass

100%|██████████| 5/5 [00:56<00:00, 11.23s/it]


When this is done, use OpenCV to vertically concatenate in groups of 20

In [11]:
group_size = 50
last_group = 0
group_count = 0
n = 0
img_arr = []
for i, outlier_name in enumerate(tqdm(df_outliers.Name)):
    
    infile = output_path.joinpath(outlier_name + '.png')
    img = cv.imread(str(infile))
    img_arr.append( img )
    group_count += 1
    
    if group_count == group_size or i == len(df_outliers[df_outliers.Unusable]) - 1:
        outfile = str(output_path.joinpath('outliers_discard_%03d..%03d.png' % (last_group + 1, i + 1)))

        concat = cv.vconcat(img_arr)
        cv.imwrite(outfile, concat)

        group_count = 0
        last_group = i + 1
        img_arr = []
        n += 1


100%|██████████| 148/148 [00:14<00:00, 10.04it/s]


Delete the individual PNGs

In [12]:
for i, outlier_name in enumerate(tqdm(df_outliers.Name)):
    
    file = output_path.joinpath(outlier_name + '.png')
    try:
        file.unlink()
    except:
        pass

100%|██████████| 148/148 [00:00<00:00, 608.54it/s]
